In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

In [ ]:
%pip install openpyxl

In [ ]:
#Load datasets
DATA_DIR = Path("datasets")
IMAGE_DIR = Path("satellite_images/train")

train_df = pd.read_excel(DATA_DIR / "train(1).xlsx")

print(train_df.shape)
train_df.head()

In [ ]:
#Basic Cleaning
train_df = train_df.drop(columns=["date", "zipcode"])

train_df["is_renovated"] = (train_df["yr_renovated"] > 0).astype(int)
train_df = train_df.drop(columns=["yr_renovated"])

In [ ]:
#Attach image paths
def image_path_from_id(pid):
    path = IMAGE_DIR / f"{pid}.png"
    return str(path) if path.exists() else None

train_df["image_path"] = train_df["id"].apply(image_path_from_id)

# Drop rows without images
train_df = train_df.dropna(subset=["image_path"])


In [ ]:
#Separate features & target
TARGET = "price"

X = train_df.drop(columns=["price", "id", "image_path"])
y = train_df["price"]


In [ ]:
# Scale tabular features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)


In [ ]:
# Final dataset
final_df = pd.concat(
    [
        train_df[["id", "image_path"]].reset_index(drop=True),
        X_scaled.reset_index(drop=True),
        y.reset_index(drop=True)
    ],
    axis=1
)

final_df.to_csv("train_with_images.csv", index=False)
final_df.head()
